### RAG PIPELINES -> DATA INGESTION TO VECTOR DB PIPELINE

In [ ]:
# Importing necesssary libraries

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
# Reading all the pdf's in the directory
def process_pdf(pdf_directory_path):
    """Process all pdf in a directory"""
    all_documents=[]
    pdf_dir = Path(pdf_directory_path)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory path")

    # processing individual files
    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            document = loader.load()

            # adding extra meta data to each of the DOC Datatype
            for doc in document:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(document)
            print(f" ✔️ loaded {len(document)} pages")

        except Exception as e:
            print(f" x error: {e}")

    print(f"\n Total pdf document pages loaded accross board is {len(all_documents)}")
    return all_documents

# using the process pdf function
all_pdf_documents = process_pdf("../data")



Found 5 PDF files in the directory path

Processing artificial_intelligence_overview.pdf
 ✔️ loaded 1 pages

Processing cloud_computing_architectures.pdf
 ✔️ loaded 1 pages

Processing cybersecurity_fundamentals.pdf
 ✔️ loaded 1 pages

Processing database_systems_and_storage.pdf
 ✔️ loaded 1 pages

Processing quantum_computing_principles.pdf
 ✔️ loaded 1 pages

 Total pdf document pages loaded accross board is 5


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\artificial_intelligence_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence_overview.pdf', 'file_type': 'pdf'}, page_content='Artificial Intelligence: Evolution and Frontiers\nIntroduction\nArtificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially\ncomputer systems. These processes include learning (the acquisition of information and rules for using the\ninformation), reasoning (using rules to reach approximate or definite conclusions), and self-correction. Over the past\nseveral decades, AI has transitioned from academic curiosity to an essential driver of modern technological\ninfrastructure.\nThe Paradigm Shift from Symbolic to Connectionist AI\nEarly artificial intelligence research focused heavily on symbolic AI, often called Good Old-Fashioned AI (GOFAI).\nSymbolic systems

In [ ]:
# Text Splitting gets into chunks -> CHUNKING

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Spliting documents into smaller chucnks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ", ""]
    )

    # using the text splitter object to split the document datatypes we had into another document datatype chunks
    splitted_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} document pages into {len(splitted_docs)} chunks")

    # showing an example of a chunnk document type
    print(f"\n Example of a chunk")
    print(f"Content: {splitted_docs[0].page_content[:200]}...")
    print(f"Metadata: {splitted_docs[0].metadata}")

    return splitted_docs

In [ ]:
chunks = split_documents(all_pdf_documents)

Splitted 5 document pages into 15 chunks

 Example of a chunk
Content: Artificial Intelligence: Evolution and Frontiers
Introduction
Artificial Intelligence (AI) represents the simulation of human intelligence processes by machines, especially
computer systems. These pro...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\artificial_intelligence_overview.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'artificial_intelligence_overview.pdf', 'file_type': 'pdf'}


### EMBEDDINGS AND VECTOR STORE DB

In [ ]:
# Importing necessary libraries
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# the embedding class
class EmbeddingManager:
    """Handles documents embedding generations using sentence transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: Hugging face model for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()



    def _load_model(self):
        """Loading the SentenceTransformer models"""
        try:
            print(f"Loading the embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded succesfully. Embedding model dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts : List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy arrays of embeddings with shape (len(texts), embeddings_dimension)
        """
        if not self.model:
            raise ValueError("Model no Loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated an embeddings with shape {embeddings.shape}")
        return embeddings



    def get_embedding_dimensions(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not Loaded")

        return self.model.get_sentence_embedding_dimension()



# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading the embedding model: all-MiniLM-L6-v2


c:\Users\JOSHUA ASEMANI\Music\AI_Chatbots\Traditional-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JOSHUA ASEMANI\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\JOSHUA ASEMANI\Music\AI_Chatbots\Traditional-RAG

Model Loaded succesfully. Embedding model dimension: 384


C:\Users\JOSHUA ASEMANI\AppData\Local\Temp\ipykernel_8396\4275882136.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded succesfully. Embedding model dimension: {self.model.get_sentence_embedding_dimension()}")


### VECTOR STORE CLASS

In [ ]:
class VectorStore:
    """Manages the documents embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist the vector store.
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()




    # Initialize store function
    def _initialize_store(self):
        """Initialize chromaDB client and collection"""

        try:
            # create a persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or Create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    # FUnction to add documents chunks and their embeddings
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Adding documents and their embeddings to the vector store

        Args:
            documents: List of langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store")

        # Prepare data for chromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = Dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)


            # Document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

            # Add to collection
            try:
                self.collection.add(
                    ids,
                    metadatas,
                    documents=documents_text,
                    embeddings=embeddings_list
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total doucments in collection {self.collection.count()}")


            except Exception as e:
                print(f"Error adding to documents to vector store: {e}")
                raise



# Initializing vector store
vector_store = VectorStore()
vector_store













Error initializing vector store: name 'os' is not defined


NameError: name 'os' is not defined